# ACIT4620 — Notebook 04: Standardized Validation Metrics & Research Figures

**Evaluation only, no GPU retraining; no test evaluation.** Attach saved outputs from E1, E2a, E2a-Long, E2a-50 and, when available, E2b. Every model must have the **same 1,441 validation image IDs, labels, and lesion IDs**. Use uncalibrated, single-view probability predictions. This notebook also exports CSV, IEEE-friendly LaTeX tables and figures.

**Caution:** older E1/E2a prediction exports may be temperature-scaled; verify whether `validation_predictions.csv` contains **uncalibrated** original-view probabilities. Otherwise regenerate these from stored validation logits/checkpoints before including them in the primary comparison. The notebook deliberately does not infer their semantics.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display
print('Attach notebook outputs using Kaggle Add Input; check files below:')
for f in sorted(Path('/kaggle/input').rglob('validation_predictions.csv')):
    print(f)

In [ ]:
"""ACIT4620 shared, validation-only, seven-class evaluation.
Load saved Notebook outputs using Kaggle Add Input. NEVER evaluate test files here.
"""
from pathlib import Path
import json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_recall_fscore_support,
    f1_score, precision_score, recall_score, confusion_matrix, roc_auc_score,
    average_precision_score, log_loss, brier_score_loss)

CLASSES = ['nv','mel','bkl','bcc','akiec','vasc','df']
K = 7
OUTPUT = Path('/kaggle/working/ACIT4620_shared_validation_report')
OUTPUT.mkdir(parents=True, exist_ok=True)

def evaluate(y, p, ids, lesions, model_name):
    y=np.asarray(y,dtype=int)
    p=np.asarray(p,dtype=float)
    ids=np.asarray(ids,dtype=str)
    lesions=np.asarray(lesions,dtype=str)
    assert p.shape==(len(y), K) and len(ids)==len(y)==len(lesions)
    assert np.isfinite(p).all() and (p>=0).all() and (p<=1).all()
    assert np.allclose(p.sum(axis=1),1,atol=1e-4), 'Probabilities must sum to 1'
    assert np.isin(y,np.arange(K)).all()
    assert len(set(ids))==len(ids), 'Duplicate image IDs'
    pred=np.argmax(p,axis=1)
    onehot=np.eye(K,dtype=int)[y]
    prec,rec,f1,sup=precision_recall_fscore_support(y,pred,labels=np.arange(K),zero_division=0)
    cm=confusion_matrix(y,pred,labels=np.arange(K))
    per=[]
    for j,c in enumerate(CLASSES):
        positive=onehot[:,j]
        if len(np.unique(positive))<2: raise ValueError(f'Class {c} has no positives or negatives')
        per.append({'model':model_name,'class':c,'support':int(sup[j]),
          'precision':float(prec[j]),'recall':float(rec[j]),'f1':float(f1[j]),
          'ovr_auroc':float(roc_auc_score(positive,p[:,j])),
          'ovr_average_precision':float(average_precision_score(positive,p[:,j]))})
    # one-vs-rest melanoma confusion matrix derived from fixed argmax decisions
    j=CLASSES.index('mel'); pos=y==j; mel_pred=pred==j
    tp=int((pos&mel_pred).sum()); fn=int((pos&~mel_pred).sum())
    fp=int((~pos&mel_pred).sum()); tn=int((~pos&~mel_pred).sum())
    summary={'model':model_name,'split':'validation','n_images':len(y),
      'n_lesions':len(set(lesions)),
      'accuracy':float(accuracy_score(y,pred)),
      'macro_f1':float(f1_score(y,pred,labels=np.arange(K),average='macro',zero_division=0)),
      'weighted_f1':float(f1_score(y,pred,labels=np.arange(K),average='weighted',zero_division=0)),
      'macro_precision':float(precision_score(y,pred,labels=np.arange(K),average='macro',zero_division=0)),
      'macro_recall':float(recall_score(y,pred,labels=np.arange(K),average='macro',zero_division=0)),
      'balanced_accuracy':float(balanced_accuracy_score(y,pred)),
      'macro_ovr_auroc':float(np.mean([d['ovr_auroc'] for d in per])),
      'macro_ovr_average_precision':float(np.mean([d['ovr_average_precision'] for d in per])),
      'nll_uncalibrated':float(log_loss(y,np.clip(p,1e-12,1),labels=np.arange(K))),
      'multiclass_brier':float(np.mean(np.sum((p-onehot)**2,axis=1))),
      'melanoma_precision':float(tp/max(1,tp+fp)),
      'melanoma_recall':float(tp/max(1,tp+fn)),
      'melanoma_specificity':float(tn/max(1,tn+fp)),
      'melanoma_f1':float(f1[j]),
      'melanoma_tp':tp,'melanoma_fn':fn,'melanoma_fp':fp,'melanoma_tn':tn,
      'melanoma_fnr':float(fn/max(1,tp+fn)),
      'melanoma_fpr':float(fp/max(1,fp+tn)),
      'melanoma_auroc':float(per[j]['ovr_auroc']),
      'melanoma_average_precision':float(per[j]['ovr_average_precision'])}
    return summary,per,cm

def read_predictions(path):
    path=Path(path)
    if path.suffix=='.npz':
        with np.load(path,allow_pickle=False) as z:
            keys=set(z.files)
            if 'y_true' not in keys or 'probs' not in keys:
                raise ValueError(f'{path}: missing y_true/probs')
            y=z['y_true']; p=z['probs']
            ids=z['image_id'] if 'image_id' in keys else None
            lesions=z['lesion_id'] if 'lesion_id' in keys else None
            # Earlier E2a exports may store calibrated original-view probability.
            # Do NOT treat them as identical to raw softmax without corroboration.
            if 'calibrated_tta_probs' not in keys and 'tta_probs' in keys:
                raise ValueError(f'{path}: earlier NPZ export may contain temperature-scaled probabilities. Attach raw logits or an explicit raw CSV and verify semantics.')
    elif path.suffix=='.csv':
        df=pd.read_csv(path)
        required=['image_id','lesion_id','true_dx']+[f'prob_{x}' for x in CLASSES]
        missing=set(required)-set(df.columns)
        if missing: raise ValueError(f'{path}: missing columns {missing}')
        y=df['true_dx'].map({c:i for i,c in enumerate(CLASSES)})
        assert y.notna().all(), 'Unknown class labels'
        ids=df.image_id.to_numpy(str); lesions=df.lesion_id.to_numpy(str)
        p=df[[f'prob_{c}' for c in CLASSES]].to_numpy(float)
    else: raise ValueError('Expected .csv or .npz')
    if ids is None or lesions is None:
        raise ValueError(f'{path}: image_id or lesion_id missing; use corresponding CSV export')
    return np.asarray(y,dtype=int), np.asarray(p,dtype=float),np.asarray(ids,dtype=str),np.asarray(lesions,dtype=str)

def locate_experiments(root=Path('/kaggle/input')):
    root=Path(root)
    found={}
    # Match output directory names as saved in original experiment configs.
    patterns={
      'E1':'*',
      'E2a':'efficientnet_b0_classweighted_e2a',
      'E2a-Long':'efficientnet_b0_classweighted_e2a_long',
      'E2a-50':'efficientnet_b0_classweighted_e2a_50epochs',
      'E2b-Rotation25':'efficientnet_b0_classweighted_e2b_rotation25'}
    for label,folder in patterns.items():
        candidates=[f for f in root.rglob('validation_predictions.csv') if
           (folder=='*' or f.parent.name==folder)]
        if label=='E1':
            candidates=[p for p in candidates if p.parent.name not in set(patterns.values()) and not p.parent.name.startswith('efficientnet_b0_classweighted')]
        candidates=sorted(set(candidates))
        if len(candidates)==1: found[label]=candidates[0]
        elif len(candidates)>1:
            print('AMBIGUOUS:',label, 'choose one saved version explicitly:',*[str(p) for p in candidates],sep='\n  ')
    return found

def run_report(paths):
    if not paths: raise ValueError('No prediction files supplied; attach saved notebook outputs through Add Input')
    summaries=[]; classes=[]; image_refs={}
    for name,path in paths.items():
        y,p,ids,lesions=read_predictions(path)
        imap={i:(int(t),l) for i,t,l in zip(ids,y,lesions)}
        if image_refs:
            ref=next(iter(image_refs.values()))
            if imap!=ref: raise AssertionError(f'{name}: different validation image IDs, diagnoses, or lesion IDs')
        image_refs[name]=imap
        s,per,cm=evaluate(y,p,ids,lesions,name)
        summaries.append(s); classes.extend(per)
        pd.DataFrame(cm,index=CLASSES,columns=CLASSES).to_csv(OUTPUT/f'{name}_confusion_matrix.csv')
        fig,ax=plt.subplots(figsize=(6,5))
        im=ax.imshow(cm,cmap='Blues'); fig.colorbar(im,ax=ax)
        ax.set_xticks(range(K),CLASSES,rotation=45,ha='right');ax.set_yticks(range(K),CLASSES)
        ax.set_xlabel('Predicted');ax.set_ylabel('True');ax.set_title(f'{name}: validation confusion matrix')
        fig.tight_layout();fig.savefig(OUTPUT/f'{name}_confusion_matrix.png',dpi=180);plt.close(fig)
    summary=pd.DataFrame(summaries).set_index('model')
    per_class=pd.DataFrame(classes)
    summary.to_csv(OUTPUT/'validation_model_metrics.csv')
    per_class.to_csv(OUTPUT/'validation_per_class_metrics.csv',index=False)
    summary.to_latex(OUTPUT/'validation_model_metrics.tex',float_format='%.4f')
    per_class.to_latex(OUTPUT/'validation_per_class_metrics.tex',index=False,float_format='%.4f')
    cols=['accuracy','macro_f1','balanced_accuracy','melanoma_recall','macro_ovr_auroc','macro_ovr_average_precision']
    ax=summary[cols].T.plot.bar(figsize=(12,5),rot=20)
    ax.set_ylabel('Score');ax.set_ylim(0,1);ax.set_title('HAM10000 validation-only model comparison')
    ax.figure.tight_layout();ax.figure.savefig(OUTPUT/'validation_model_comparison.png',dpi=180);plt.close(ax.figure)
    (OUTPUT/'evaluation_metadata.json').write_text(json.dumps({
       'scope':'VALIDATION ONLY; test not evaluated',
       'selection_metric':'macro_f1', 'class_order':CLASSES,
       'probs':'original-view, uncalibrated softmax; verify older E1/E2a exports',
       'paths':{k:str(v) for k,v in paths.items()},
       'limitations':'Single-seed metrics, model selection reuse; no independent test or CIs inferred'},indent=2))
    print('Results saved to:',OUTPUT)
    print(summary[cols].round(4).to_string())
    return summary,per_class


## Input selection

The automatic discovery is a convenience; **verify** each path maps to the right saved Kaggle version, and omit any export whose probability calibration is unclear. E1 may have a different folder structure; set paths manually if needed. `validation_predictions.csv` must contain columns `image_id`, `lesion_id`, `true_dx`, and `prob_nv`, `prob_mel`, etc.

In [ ]:
found=locate_experiments()
for k,p in found.items(): print(k,':',p)
# EDIT paths after checking them if necessary, then run next cell.
PATHS=found.copy()
assert PATHS, 'Attach saved Kaggle notebook outputs and inspect the printed paths' 

## Strict validation comparison

This cell fails if the attached predictions differ in image IDs, lesion assignments, or true labels. It saves confusion matrices, AUROC and AP, melanoma TP/FP/TN/FN, macro and weighted F1, macro precision, NLL and Brier score. **Do not attach any test predictions** here.

In [ ]:
summary, per_class = run_report(PATHS)
display(summary)
display(per_class)
print('REPORT DIRECTORY:',OUTPUT)

## Interpretation and next steps

Use these **validation-only** metrics as the working evidence tables. Do not call single-seed improvements statistically significant. Add E2b rotation-ablation results after that notebook finishes, then use a validation-selected CNN checkpoint to prepare E3 MLP and E4 ANFIS with matched frozen 1,280-D features. Run repeated seeds, XAI, efficiency, and locked test only after decisions are finalized.